# 🎬 YouTube Shorts 動画自動生成

**使い方**
1. 「ランタイム」→「すべてのセルを実行」
2. Googleドライブのマウント許可をクリック
3. 完成した動画はドライブの `YouTube_Production/プロジェクトフォルダ/output/` に保存されます

**事前に必要なこと**
- Google Apps Script（GAS）の `runPipelineStandalone` を実行済みであること
- Google Drive に `YouTube_Production/` フォルダとプロジェクトフォルダが存在すること

In [ ]:
# ── セル1: 依存パッケージのインストール ──────────────────────────────────────
!pip install -q gtts requests
!apt-get install -q -y ffmpeg
print('✅ インストール完了')

In [ ]:
# ── セル2: Googleドライブをマウント ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ ドライブ接続完了')

In [ ]:
# ── セル3: プロジェクトフォルダを自動検出 ───────────────────────────────────
import glob, os

PRODUCTION_DIR = '/content/drive/MyDrive/YouTube_Production'

# 最新のプロジェクトフォルダを自動選択
folders = sorted(
    [f for f in glob.glob(f'{PRODUCTION_DIR}/*/') if os.path.isdir(f)],
    key=os.path.getmtime,
    reverse=True
)

if not folders:
    raise FileNotFoundError(
        f'プロジェクトフォルダが見つかりません。\n'
        f'GASの runPipelineStandalone を先に実行してください。\n'
        f'検索場所: {PRODUCTION_DIR}'
    )

PROJECT_FOLDER = folders[0].rstrip('/')

print('検出されたプロジェクトフォルダ一覧:')
for i, f in enumerate(folders[:5]):
    mark = '← 使用' if i == 0 else ''
    print(f'  {i+1}. {os.path.basename(f.rstrip("/"))} {mark}')

print(f'\n✅ 使用フォルダ: {os.path.basename(PROJECT_FOLDER)}')

In [ ]:
# ── セル4（任意）: 別のフォルダを使いたい場合は手動で指定 ───────────────────
# 上のセル3で自動選択したフォルダを使う場合はこのセルをスキップしてください

# PROJECT_FOLDER = '/content/drive/MyDrive/YouTube_Production/20250525_123456_テーマ名'

print(f'現在のプロジェクト: {PROJECT_FOLDER}')

In [ ]:
# ── セル5: make_video.py をドライブからコピー ────────────────────────────────
import shutil

SCRIPT_IN_DRIVE = '/content/drive/MyDrive/ans-trainer/make_video.py'
SCRIPT_LOCAL    = '/content/make_video.py'

if os.path.exists(SCRIPT_IN_DRIVE):
    shutil.copy(SCRIPT_IN_DRIVE, SCRIPT_LOCAL)
    print(f'✅ make_video.py をドライブからコピーしました')
else:
    # ドライブにない場合はGitHubからダウンロード
    !wget -q -O /content/make_video.py \
        https://raw.githubusercontent.com/hayashi0821kousuke-maker/ans-trainer/claude/happy-darwin-snpLH/make_video.py
    print('✅ make_video.py をGitHubからダウンロードしました')

In [ ]:
# ── セル6: 動画を生成 ────────────────────────────────────────────────────────
# gTTSを使って音声生成 → ケン・バーンズ動画クリップ → 最終合成

!python /content/make_video.py \
    --project "{PROJECT_FOLDER}" \
    --tts gtts

print('\n動画生成が完了しました！')

In [ ]:
# ── セル7: 完成動画を表示 ────────────────────────────────────────────────────
from IPython.display import Video, display
import glob

output_files = sorted(
    glob.glob(f'{PROJECT_FOLDER}/output/shorts_*.mp4'),
    key=os.path.getmtime,
    reverse=True
)

if output_files:
    latest = output_files[0]
    size_mb = os.path.getsize(latest) / 1_048_576
    print(f'✅ 完成動画: {os.path.basename(latest)}')
    print(f'   サイズ: {size_mb:.1f} MB')
    print(f'   保存先: {latest}')
    display(Video(latest, width=360))
else:
    print('⚠ 動画ファイルが見つかりません。セル6を再実行してください。')